In [1]:
import gc
import os
from collections import Counter

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter

from dataset.pac_matrix import PAC5MatrixHLGapTransferLearningDataset
from gcnn.base_model import GroupEquivariantCNN

K_FOLDS = 5
BATCH_SIZE = 128
PRETRAINED_WEIGHTS_PATH = (
    "../models/gcnn_3.7M-pac_grid-[PAC5-1-no_high_fidelity]-fold_1/best.ckpt"
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Stratify folds on 0.5 eV bins of CAM-B3LYP HL-gap
# Merge bins with fewer than MINIMUM_MEMBERS_PER_LABEL molecules into smaller neighbour
BAND_GAP_BIN_WIDTH = 0.5
MINIMUM_MEMBERS_PER_LABEL = 10


def merge_sparse_bins(bin_indices, minimum_members):
    merged = np.asarray(bin_indices).copy()

    while True:
        counts = Counter(merged.tolist())
        ordered_bins = sorted(counts)
        if len(ordered_bins) <= 1:
            return merged

        sparse_bins = [
            bin_id for bin_id in ordered_bins if counts[bin_id] < minimum_members
        ]
        if not sparse_bins:
            return merged

        smallest_bin = min(sparse_bins, key=lambda bin_id: counts[bin_id])
        position = ordered_bins.index(smallest_bin)

        if position == 0:
            neighbour = ordered_bins[1]
        elif position == len(ordered_bins) - 1:
            neighbour = ordered_bins[position - 1]
        else:
            left_neighbour = ordered_bins[position - 1]
            right_neighbour = ordered_bins[position + 1]
            neighbour = (
                left_neighbour
                if counts[left_neighbour] <= counts[right_neighbour]
                else right_neighbour
            )

        merged[merged == smallest_bin] = neighbour


def build_band_gap_bin_labels(dataset):
    band_gaps = np.array([item[2] for item in dataset.paired_data], dtype=float)

    lowest_edge = np.floor(np.min(band_gaps) / BAND_GAP_BIN_WIDTH) * BAND_GAP_BIN_WIDTH
    highest_edge = np.ceil(np.max(band_gaps) / BAND_GAP_BIN_WIDTH) * BAND_GAP_BIN_WIDTH
    edges = np.arange(
        lowest_edge, highest_edge + BAND_GAP_BIN_WIDTH / 2, BAND_GAP_BIN_WIDTH
    )

    bin_indices = np.digitize(band_gaps, edges[1:-1], right=False)
    bin_indices = merge_sparse_bins(bin_indices, MINIMUM_MEMBERS_PER_LABEL)

    remapped = {
        bin_id: position
        for position, bin_id in enumerate(sorted(set(bin_indices.tolist())))
    }
    return np.array([f"gap_bin_{remapped[bin_id]:02d}" for bin_id in bin_indices])


class GCNNDeltaCorrection(nn.Module):
    def __init__(self, kernel_size=5, num_hidden=5, hidden_channels=48):
        super().__init__()

        self.feature_extractor = GroupEquivariantCNN(
            property_types=["intensive"],
            kernel_size=kernel_size,
            num_hidden=num_hidden,
            hidden_channels=hidden_channels,
        )
        self.feature_extractor.thermodynamic_head = nn.Identity()
        self.feature_extractor.electronic_head = nn.Identity()

        self.correction_head = nn.Sequential(
            nn.BatchNorm1d(hidden_channels),
            nn.Linear(hidden_channels, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 1),
        )

    def forward(self, pac_tensor, low_level_bg):
        intensive_features, _ = self.feature_extractor.extract_features(pac_tensor)
        structural_features = intensive_features
        predicted_bg = self.correction_head(structural_features)
        return predicted_bg, torch.zeros_like(predicted_bg)


dataset = PAC5MatrixHLGapTransferLearningDataset()

stratify_labels = build_band_gap_bin_labels(dataset)
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold_results = []

Using device: cuda
Loading high-level theory (CAM-B3LYP) dataset...
Fetching PM7 values...
Matched 5100 paired samples.


In [2]:
for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), stratify_labels), 1
):
    print(f"\n======[ Fold {fold}/{K_FOLDS} ]======")

    try:
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=0.125,
            random_state=42,
            stratify=stratify_labels[train_val_idx],
        )
    except ValueError:
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.125, random_state=42
        )

    log_dir = f"../logs/gcnn_3.7M-pac_grid-[PAC5-hl_gap-to_camb3lyp_hl_gap]-finetune/fold_{fold}"
    os.makedirs(log_dir, exist_ok=True)
    writer = SummaryWriter(log_dir=log_dir)

    train_sub = Subset(dataset, train_idx)
    val_sub = Subset(dataset, val_idx)
    test_sub = Subset(dataset, test_idx)

    train_loader = DataLoader(
        train_sub, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True
    )
    val_loader = DataLoader(
        val_sub, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True
    )
    test_loader = DataLoader(
        test_sub, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True
    )

    model = GCNNDeltaCorrection().to(device)

    pretrained_weights = torch.load(PRETRAINED_WEIGHTS_PATH, map_location=device)
    state_dict = pretrained_weights.get("state_dict", pretrained_weights)
    state_dict = {
        k.replace("model.", "").replace("feature_extractor.", ""): v
        for k, v in state_dict.items()
    }

    model.feature_extractor.load_state_dict(state_dict, strict=False)

    criterion = nn.MSELoss()

    optimizer = optim.AdamW(
        [
            {"params": model.feature_extractor.parameters(), "lr": 5e-4},
            {"params": model.correction_head.parameters(), "lr": 5e-3},
        ],
        weight_decay=1e-4,
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[5e-4, 5e-3], steps_per_epoch=len(train_loader), epochs=100
    )

    save_dir = "../models/gcnn_3.7M-pac_grid-[PAC5-hl_gap-to_camb3lyp_hl_gap]-finetune"
    os.makedirs(save_dir, exist_ok=True)
    save_path = f"{save_dir}/fold_{fold}.pth"

    best_val_loss = float("inf")

    for epoch in range(1, 101):
        model.train()
        running_loss = 0.0

        for pac_tensor, low_bg, high_bg, _, _ in train_loader:
            pac_tensor = pac_tensor.to(device)
            low_bg = low_bg.to(device)
            high_bg = high_bg.to(device)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=device.type == "cuda",
            ):
                corrected_bg, _ = model(pac_tensor, low_bg)
                loss = criterion(corrected_bg, high_bg)

            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)

        model.eval()
        avg_val_loss = 0.0

        with torch.no_grad():
            for pac_tensor, low_bg, high_bg, _, _ in val_loader:
                pac_tensor = pac_tensor.to(device)
                low_bg = low_bg.to(device)
                high_bg = high_bg.to(device)

                with torch.autocast(
                    device_type=device.type,
                    dtype=torch.bfloat16,
                    enabled=device.type == "cuda",
                ):
                    corrected_bg, _ = model(pac_tensor, low_bg)
                    avg_val_loss += criterion(corrected_bg, high_bg).item()

        avg_val_loss /= len(val_loader)

        base_lr = optimizer.param_groups[0]["lr"]
        head_lr = optimizer.param_groups[1]["lr"]

        writer.add_scalar("loss/train", avg_train_loss, epoch)
        writer.add_scalar("loss/val", avg_val_loss, epoch)
        writer.add_scalar("hyperparameters/base_lr", base_lr, epoch)
        writer.add_scalar("hyperparameters/head_lr", head_lr, epoch)
        writer.flush()

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            marker = " (best)"
            torch.save(model.state_dict(), save_path)
        else:
            marker = ""

        if epoch % 10 == 0 or epoch == 1 or marker:
            print(
                f"Epoch {epoch:03d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}{marker}"
            )

    print(f"Best Validation Loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)
    writer.close()

    del train_sub, val_sub, test_sub
    del train_loader, val_loader, test_loader
    del model, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Loss per fold: {fold_results}")
print(f"Average Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")


======[ Fold 1/5 ]======
Epoch 001 | Train Loss: 44.9145 | Val Loss: 43.2361 (best)
Epoch 002 | Train Loss: 39.6988 | Val Loss: 36.1371 (best)
Epoch 003 | Train Loss: 28.3788 | Val Loss: 19.4869 (best)
Epoch 004 | Train Loss: 10.3354 | Val Loss: 6.5397 (best)
Epoch 005 | Train Loss: 3.9280 | Val Loss: 2.2133 (best)
Epoch 006 | Train Loss: 1.6419 | Val Loss: 0.7014 (best)
Epoch 007 | Train Loss: 0.7564 | Val Loss: 0.4765 (best)
Epoch 008 | Train Loss: 0.4781 | Val Loss: 0.2216 (best)
Epoch 009 | Train Loss: 0.3265 | Val Loss: 0.1717 (best)
Epoch 010 | Train Loss: 0.2809 | Val Loss: 0.1390 (best)
Epoch 012 | Train Loss: 0.2738 | Val Loss: 0.1261 (best)
Epoch 013 | Train Loss: 0.1881 | Val Loss: 0.1005 (best)
Epoch 014 | Train Loss: 0.1596 | Val Loss: 0.0827 (best)
Epoch 015 | Train Loss: 0.1483 | Val Loss: 0.0748 (best)
Epoch 017 | Train Loss: 0.1388 | Val Loss: 0.0723 (best)
Epoch 020 | Train Loss: 0.0694 | Val Loss: 0.0697 (best)
Epoch 023 | Train Loss: 0.0975 | Val Loss: 0.0617 (best

In [3]:
import os

import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset

from dataset.pac_matrix import PAC5MatrixHLGapTransferLearningDataset


def evaluate_bandgap_model(model, dataloader, device):
    model.eval()
    all_targets = []
    all_preds = []

    with torch.no_grad():
        for pac_tensor, low_bg, high_bg, _, _ in dataloader:
            pac_tensor = pac_tensor.to(device)
            low_bg = low_bg.to(device)

            corrected_bg, _ = model(pac_tensor, low_bg)

            all_preds.extend(corrected_bg.cpu().numpy())
            all_targets.extend(high_bg.numpy())

    all_preds = np.array(all_preds).flatten()
    all_targets = np.array(all_targets).flatten()

    errors = all_preds - all_targets
    abs_errors = np.abs(errors)

    mae = np.mean(abs_errors)
    rmsd = np.sqrt(np.mean(errors**2))
    max_err = np.max(abs_errors)
    min_err = np.min(abs_errors)
    r2 = r2_score(all_targets, all_preds)

    return mae, rmsd, max_err, min_err, r2


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluating on device: {device}")

dataset = PAC5MatrixHLGapTransferLearningDataset()

k_folds = 5
safe_stratify_labels = build_band_gap_bin_labels(dataset)

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

fold_train_metrics = []
fold_val_metrics = []
fold_test_metrics = []

for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), safe_stratify_labels), 1
):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    try:
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=0.125,
            random_state=42,
            stratify=safe_stratify_labels[train_val_idx],
        )
    except ValueError:
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.125, random_state=42
        )

    train_sub = Subset(dataset, train_idx)
    val_sub = Subset(dataset, val_idx)
    test_sub = Subset(dataset, test_idx)

    train_loader = DataLoader(train_sub, batch_size=256, shuffle=False)
    val_loader = DataLoader(val_sub, batch_size=256, shuffle=False)
    test_loader = DataLoader(test_sub, batch_size=256, shuffle=False)

    model = GCNNDeltaCorrection().to(device)

    load_path = f"../models/gcnn_3.7M-pac_grid-[PAC5-hl_gap-to_camb3lyp_hl_gap]-finetune/fold_{fold}.pth"
    try:
        model.load_state_dict(torch.load(load_path, map_location=device))
    except FileNotFoundError:
        print(f"Could not find {load_path}. Skipping this fold.")
        continue

    train_results = evaluate_bandgap_model(model, train_loader, device)
    val_results = evaluate_bandgap_model(model, val_loader, device)
    test_results = evaluate_bandgap_model(model, test_loader, device)

    fold_train_metrics.append(train_results)
    fold_val_metrics.append(val_results)
    fold_test_metrics.append(test_results)

    t_mae, t_rmsd, t_max, t_min, t_r2 = train_results
    print(
        f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
    )

    v_mae, v_rmsd, v_max, v_min, v_r2 = val_results
    print(
        f" + Val -> MAE: {v_mae:.4f}\tRMSD: {v_rmsd:.4f}\tMax: {v_max:.4f}\tMin: {v_min:.4f}\tR^2: {v_r2:.4f}"
    )

    te_mae, te_rmsd, te_max, te_min, te_r2 = test_results
    print(
        f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
    )

if fold_train_metrics and fold_val_metrics and fold_test_metrics:
    print("\n======[ Final 5-fold cross validation results ]======")

    train_metrics_array = np.array(fold_train_metrics)
    val_metrics_array = np.array(fold_val_metrics)
    test_metrics_array = np.array(fold_test_metrics)

    avg_train = np.mean(train_metrics_array, axis=0)
    std_train = np.std(train_metrics_array, axis=0, ddof=1)

    avg_val = np.mean(val_metrics_array, axis=0)
    std_val = np.std(val_metrics_array, axis=0, ddof=1)

    avg_test = np.mean(test_metrics_array, axis=0)
    std_test = np.std(test_metrics_array, axis=0, ddof=1)

    print("Train:")
    print(f" + MAE: {avg_train[0]:.4f} ± {std_train[0]:.4f}")
    print(f" + RMSD: {avg_train[1]:.4f} ± {std_train[1]:.4f}")
    print(f" + Max: {avg_train[2]:.4f} ± {std_train[2]:.4f}")
    print(f" + Min: {avg_train[3]:.4f} ± {std_train[3]:.4f}")
    print(f" + R^2: {avg_train[4]:.4f} ± {std_train[4]:.4f}\n")

    print("Validation:")
    print(f" + MAE: {avg_val[0]:.4f} ± {std_val[0]:.4f}")
    print(f" + RMSD: {avg_val[1]:.4f} ± {std_val[1]:.4f}")
    print(f" + Max: {avg_val[2]:.4f} ± {std_val[2]:.4f}")
    print(f" + Min: {avg_val[3]:.4f} ± {std_val[3]:.4f}")
    print(f" + R^2: {avg_val[4]:.4f} ± {std_val[4]:.4f}\n")

    print("Test:")
    print(f" + MAE: {avg_test[0]:.4f} ± {std_test[0]:.4f}")
    print(f" + RMSD: {avg_test[1]:.4f} ± {std_test[1]:.4f}")
    print(f" + Max: {avg_test[2]:.4f} ± {std_test[2]:.4f}")
    print(f" + Min: {avg_test[3]:.4f} ± {std_test[3]:.4f}")
    print(f" + R^2: {avg_test[4]:.4f} ± {std_test[4]:.4f}")

Evaluating on device: cuda
Loading high-level theory (CAM-B3LYP) dataset...
Fetching PM7 values...
Matched 5100 paired samples.

======[ Evaluating fold 1/5 ]======
 + Train -> MAE: 0.0278	RMSD: 0.0371	Max: 0.2635	Min: 0.0000	R^2: 0.9988
 + Val -> MAE: 0.1021	RMSD: 0.1575	Max: 0.8739	Min: 0.0001	R^2: 0.9789
 + Test -> MAE: 0.0987	RMSD: 0.1814	Max: 2.7896	Min: 0.0002	R^2: 0.9711

======[ Evaluating fold 2/5 ]======
 + Train -> MAE: 0.0238	RMSD: 0.0313	Max: 0.2950	Min: 0.0000	R^2: 0.9991
 + Val -> MAE: 0.0976	RMSD: 0.1406	Max: 0.7949	Min: 0.0001	R^2: 0.9821
 + Test -> MAE: 0.0959	RMSD: 0.1517	Max: 1.1594	Min: 0.0001	R^2: 0.9800

======[ Evaluating fold 3/5 ]======
 + Train -> MAE: 0.0276	RMSD: 0.0362	Max: 0.1581	Min: 0.0000	R^2: 0.9988
 + Val -> MAE: 0.0958	RMSD: 0.1418	Max: 0.7620	Min: 0.0001	R^2: 0.9820
 + Test -> MAE: 0.1048	RMSD: 0.1608	Max: 1.0124	Min: 0.0001	R^2: 0.9769

======[ Evaluating fold 4/5 ]======
 + Train -> MAE: 0.0235	RMSD: 0.0308	Max: 0.1623	Min: 0.0000	R^2: 0.9992
 + 

In [4]:
import os

import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset

from dataset.pac_matrix import PAC5MatrixHLGapTransferLearningDataset

TARGET_FOLD = 2
K_FOLDS = 5
BATCH_SIZE = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset = PAC5MatrixHLGapTransferLearningDataset()

safe_stratify_labels = build_band_gap_bin_labels(dataset)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

target_train_idx, target_val_idx, target_test_idx = None, None, None

for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), safe_stratify_labels), 1
):
    if fold == TARGET_FOLD:
        try:
            train_idx, val_idx = train_test_split(
                train_val_idx,
                test_size=0.125,
                random_state=42,
                stratify=safe_stratify_labels[train_val_idx],
            )
        except ValueError:
            train_idx, val_idx = train_test_split(
                train_val_idx, test_size=0.125, random_state=42
            )

        target_train_idx = train_idx
        target_val_idx = val_idx
        target_test_idx = test_idx
        break

if target_train_idx is None or target_val_idx is None or target_test_idx is None:
    raise ValueError(f"Fold {TARGET_FOLD} is out of bounds for K={K_FOLDS}")

train_sub = Subset(dataset, target_train_idx)
val_sub = Subset(dataset, target_val_idx)
test_sub = Subset(dataset, target_test_idx)

train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_sub, batch_size=BATCH_SIZE, shuffle=False)

model = GCNNDeltaCorrection().to(device)

model_path = f"../models/gcnn_3.7M-pac_grid-[PAC5-hl_gap-to_camb3lyp_hl_gap]-finetune/fold_{TARGET_FOLD}.pth"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"No trained model weights at {model_path} for this fold")

model.load_state_dict(torch.load(model_path, map_location=device))


def evaluate_and_plot(model, loader, split_name, fold_num, device):
    model.eval()
    results = []

    with torch.no_grad():
        for pac_tensor, low_bg, high_bg, smiles_batch, level_batch in loader:
            pac_tensor = pac_tensor.to(device)
            low_bg = low_bg.to(device)

            corrected_bg, _ = model(pac_tensor, low_bg)

            for i in range(len(smiles_batch)):
                results.append(
                    {
                        "split": split_name,
                        "smiles": smiles_batch[i],
                        "level": level_batch[i],
                        "pm7_bg": low_bg[i].item(),
                        "camb3lyp_bg": high_bg[i].item(),
                        "corrected_bg": corrected_bg[i].item(),
                    }
                )

    df_results = pd.DataFrame(results)

    y_true = df_results["camb3lyp_bg"]
    y_orig = df_results["pm7_bg"]
    y_corr = df_results["corrected_bg"]

    orig_mae = (y_orig - y_true).abs().mean()
    orig_rmsd = np.sqrt(((y_orig - y_true) ** 2).mean())
    orig_r2 = r2_score(y_true, y_orig)

    corr_mae = (y_corr - y_true).abs().mean()
    corr_rmsd = np.sqrt(((y_corr - y_true) ** 2).mean())
    corr_r2 = r2_score(y_true, y_corr)

    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1)

    levels = df_results["level"].unique()
    colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A", "#19D3F3"]

    def add_scatter_traces(col_name, subplot_col):
        for i, level in enumerate(levels):
            df_sub = df_results[df_results["level"] == level]
            fig.add_trace(
                go.Scatter(
                    x=df_sub["camb3lyp_bg"],
                    y=df_sub[col_name],
                    mode="markers",
                    name=str(level),
                    marker=dict(color=colors[i % len(colors)]),
                    text=df_sub["smiles"],
                    hovertemplate="SMILES: %{text}<br>CAM-B3LYP: %{x:.4f} eV<br>Y: %{y:.4f} eV<extra></extra>",
                    showlegend=(subplot_col == 1),
                ),
                row=1,
                col=subplot_col,
            )

    add_scatter_traces("pm7_bg", 1)
    add_scatter_traces("corrected_bg", 2)

    min_val = min(y_true.min(), y_orig.min(), y_corr.min()) - 0.5
    max_val = max(y_true.max(), y_orig.max(), y_corr.max()) + 0.5

    for col in [1, 2]:
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                line=dict(dash="dash", color="black", width=1),
                name="Parity",
                showlegend=False,
            ),
            row=1,
            col=col,
        )

    fig.update_layout(width=1100, height=550, template="plotly", font=dict(size=18))

    def get_stats_text(mae, rmsd, r2):
        return f"MAE: {mae:.3f} eV<br>RMSD: {rmsd:.3f} eV<br>R<sup>2</sup>: {r2:.3f}"

    fig.add_annotation(
        text=get_stats_text(orig_mae, orig_rmsd, orig_r2),
        xref="x domain",
        yref="y domain",
        x=0.05,
        y=0.95,
        showarrow=False,
        align="left",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
    )

    fig.add_annotation(
        text=get_stats_text(corr_mae, corr_rmsd, corr_r2),
        xref="x2 domain",
        yref="y2 domain",
        x=0.05,
        y=0.95,
        showarrow=False,
        align="left",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
    )

    fig.update_xaxes(
        title_text="<b>CAM-B3LYP HL-Gap (eV)</b>",
        row=1,
        col=1,
        range=[min_val, max_val],
    )
    fig.update_xaxes(
        title_text="<b>CAM-B3LYP HL-Gap (eV)</b>",
        row=1,
        col=2,
        range=[min_val, max_val],
    )
    fig.update_yaxes(
        title_text="<b>PM7 HL-Gap (eV)</b>", row=1, col=1, range=[min_val, max_val]
    )
    fig.update_yaxes(
        title_text="<b>Corrected HL-Gap (eV)</b>",
        row=1,
        col=2,
        range=[min_val, max_val],
    )
    fig.update_xaxes(
        linewidth=1, linecolor="black", mirror=True, ticks="inside", showline=True
    )
    fig.update_yaxes(
        linewidth=1, linecolor="black", mirror=True, ticks="inside", showline=True
    )
    fig.update_traces(marker=dict(size=10))
    fig.update_layout(
        template="plotly_white",
        font=dict(family="Helvetica", size=20),
        width=1200,
        height=600,
        paper_bgcolor="rgba(0,0,0,0)",
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.2,
            xanchor="center",
            x=0.5,
            bordercolor="black",
            borderwidth=1,
        ),
    )
    fig.update_layout(showlegend=False)

    filename = f"./bg_delta_results_finetune_fold_{fold_num}_{split_name.lower()}.svg"
    fig.write_image(filename)
    print(f"Saved {filename}")
    fig.show()

    return df_results


evaluate_and_plot(model, train_loader, "Train", TARGET_FOLD, device)
evaluate_and_plot(model, val_loader, "Val", TARGET_FOLD, device)
evaluate_and_plot(model, test_loader, "Test", TARGET_FOLD, device)

Using device: cuda
Loading high-level theory (CAM-B3LYP) dataset...


Fetching PM7 values...


Matched 5100 paired samples.


Saved ./bg_delta_results_finetune_fold_2_train.svg


Saved ./bg_delta_results_finetune_fold_2_val.svg


Saved ./bg_delta_results_finetune_fold_2_test.svg


,split,smiles,level,pm7_bg,camb3lyp_bg,corrected_bg
0,Test,c1ccc2ccccc2c1,10000_2,8.405053,7.289658,7.337432
1,Test,O=C1C=CC(=O)c2cnccc21,10101_2,8.428454,6.599849,6.696214
2,Test,N#Cc1cccc2ccccc12,10001_2,8.128585,6.908699,6.855659
3,Test,Cc1cccc2ccccc12,10001_2,8.299473,7.166934,7.176569
4,Test,Clc1ccc2ccccc2c1,10001_2,8.303826,7.234691,7.229323
...,...,...,...,...,...,...
1015,Test,c1ccc2c(c1)ccc1cc3cc4c(cc3cc12)c1cc2cccc3ccc5c...,10000_11,6.129909,4.652059,4.527915
1016,Test,c1ccc2c(c1)ccc1c2cc2cc3ccc4cccc5cc6c7ccc8ccccc...,10000_10,6.009090,4.613418,4.585098
1017,Test,c1ccc2c(c1)ccc1c2c2ccccc2c2c3ccc4ccccc4c3c3c4c...,10000_9,7.049654,5.827046,5.878554
1018,Test,c1cc2cccc3c4cc5cc6c(cc5cc4cc(c1)c23)cc1cc2c3cc...,10000_13,4.631378,2.550795,3.043156


In [5]:
df_train = evaluate_and_plot(model, train_loader, "Train", TARGET_FOLD, device)
df_val = evaluate_and_plot(model, val_loader, "Val", TARGET_FOLD, device)
df_test = evaluate_and_plot(model, test_loader, "Test", TARGET_FOLD, device)

df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)

df_all["abs_error_eV"] = (df_all["corrected_bg"] - df_all["camb3lyp_bg"]).abs()
df_all["residual_eV"] = df_all["corrected_bg"] - df_all["camb3lyp_bg"]
df_sorted = df_all.sort_values(by="abs_error_eV", ascending=False).reset_index(
    drop=True
)

cols_order = [
    "split",
    "level",
    "smiles",
    "abs_error_eV",
    "residual_eV",
    "camb3lyp_bg",
    "corrected_bg",
    "pm7_bg",
]
df_sorted = df_sorted[cols_order]


df_sorted

Saved ./bg_delta_results_finetune_fold_2_train.svg


Saved ./bg_delta_results_finetune_fold_2_val.svg


Saved ./bg_delta_results_finetune_fold_2_test.svg


,split,level,smiles,abs_error_eV,residual_eV,camb3lyp_bg,corrected_bg,pm7_bg
0,Val,10000_13,c1ccc2c(c1)cc1cc3cc4cc5c(cc4cc3c3cccc2c13)c1cc...,1.831900,1.831900,1.040019,2.871919,3.798165
1,Test,11010_5,C1=C(c2ccccc2)C=c2c1cc1cccc3cccc2c31,1.154938,-1.154938,4.945669,3.790731,6.732913
2,Test,11111_8,O=c1c2c(-c3cc(-c4ccccc4)[nH]c4nn(-c5ccccc5)c(=...,1.027022,1.027022,4.780224,5.807246,6.367736
3,Test,11000_9,c1cc2c(c1)c1c3cccc3c3ccc4ccc5ccc6ccc2c2c6c5c4c...,0.975677,0.975677,3.346456,4.322134,5.360099
4,Val,10000_13,c1ccc2cc3cc4cc5cc6cc7c(cc6cc5cc4cc3cc2c1)cc1cc...,0.880417,0.880417,1.595676,2.476093,4.652875
...,...,...,...,...,...,...,...,...
5095,Train,10101_2,Fc1ccc(Cl)c2cnccc12,0.000037,0.000037,7.060810,7.060847,8.146817
5096,Train,11011_5,Cc1ccc(-c2ccc3c4c(cccc24)-c2ccccc2-3)cc1,0.000035,0.000035,6.126643,6.126678,7.568847
5097,Train,11000_12,c1cc2ccc3ccc4cc5c6c7c8c-5ccc5ccc9ccc%10ccc(c%1...,0.000022,-0.000022,4.547022,4.547000,6.014261
5098,Train,11101_2,Clc1cccc2c1c(Cl)cn2Cl,0.000019,0.000019,7.072511,7.072530,7.373469
